# Дообучение модели RuBERT

In [ ]:
!pip install -q transformers datasets accelerate evaluate

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import evaluate
from sklearn.metrics import classification_report

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00


In [ ]:
#Подключаем датасеты
train_df = pd.read_csv("train_genre.csv")
test_df = pd.read_csv("test_genre.csv")

In [ ]:
#Перемешиваем строки
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)

## Mapping

In [ ]:
unique_labels = sorted(train_df["label"].unique())

label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}

train_df["label"] = train_df["label"].map(label2id)
test_df["label"] = test_df["label"].map(label2id)

num_labels = len(label2id)

print(label2id)

{'баллада': 0, 'загадка': 1, 'притча': 2, 'сонета': 3, 'хокку': 4, 'частушка': 5}


In [ ]:
train_dataset = Dataset.from_pandas(train_df[["text", "label"]])
test_dataset = Dataset.from_pandas(test_df[["text", "label"]])

## Токенизация

In [ ]:
model_name = "DeepPavlov/rubert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

## Модель

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you

## Общие метрики

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"],
    }

## Обучение модели

In [ ]:
training_args = TrainingArguments(
    output_dir="./rubert-genre-classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=200,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


## Оценка по классам

In [ ]:
predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

report = classification_report(y_true, y_pred, output_dict=True)
df_metrics = pd.DataFrame(report).transpose()

#Убираем aggregate строки
df_metrics = df_metrics[~df_metrics.index.isin(["accuracy", "macro avg", "weighted avg"])]

#Показатели accuracy по каждому классу
accuracy_per_class = []

for label in sorted(set(y_true)):
    idx = np.where(y_true == label)[0]
    acc_class = np.mean(y_pred[idx] == y_true[idx])
    accuracy_per_class.append(acc_class)

df_metrics["accuracy"] = accuracy_per_class

#Производим замену индексов на названия жанров
df_metrics.index = [id2label[int(i)] for i in df_metrics.index]

print(df_metrics)

## Графическое представление результатов

In [ ]:
labels = df_metrics.index
x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(10, 6))

plt.bar(x - width/2, df_metrics["accuracy"], width, label="Accuracy")
plt.bar(x + width/2, df_metrics["f1-score"], width, label="F1-score")

plt.xticks(x, labels, rotation=20)
plt.xlabel("Жанры")
plt.ylabel("Score")
plt.title("Метрики по жанрам")
plt.legend()

plt.show()

## Сохраняем результаты

In [ ]:
df_metrics.to_csv("metrics_per_class.csv")

# Интерактивное определение жанра

In [ ]:
import torch

while True:
    text = input("Введите текст стихотворения (или 'exit', если хотите выйти):\n")
    if text.lower() == "exit":
        break

    #Токенизация
    inputs = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=128,
        return_tensors="pt"
    )

    #Предсказание от модели
    with torch.no_grad():
        outputs = trainer.model(**inputs)
        logits = outputs.logits
        predicted_id = int(torch.argmax(logits, dim=1).item())
        predicted_label = id2label[predicted_id]

    print(f"Я полагаю, это {predicted_label}\n")

# Генерация стихотворений с применением дообученной RuBERT

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

AutoTokenizer.from_pretrained("sberbank-ai/rugpt3small_based_on_gpt2")
AutoModelForCausalLM.from_pretrained("sberbank-ai/rugpt3small_based_on_gpt2")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cpu"

#Генератор
gen_model_name = "sberbank-ai/rugpt3small_based_on_gpt2"
gen_tokenizer = AutoTokenizer.from_pretrained(gen_model_name)
gen_model = AutoModelForCausalLM.from_pretrained(gen_model_name).to(device)
gen_model.eval()

#Классификатор
classifier = trainer.model.to(device)
classifier.eval()

available_genres = list(label2id.keys())

def generate_4_lines_poem(genre, topic, n=5, max_new_tokens=150):
    prompt = f"Сочини законченное стихотворение из 4 строк в жанре {genre} по теме {topic}:\n"
    input_ids = gen_tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    candidates = []

    for _ in range(n):
        outputs = gen_model.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=1.0,
            eos_token_id=gen_tokenizer.eos_token_id
        )
        poem = gen_tokenizer.decode(outputs[0], skip_special_tokens=True)
        poem = poem[len(prompt):].strip()
        candidates.append(poem)

    #Выбираем лучший вариант с помощью RuBERT
    best_poem = candidates[0]
    best_score = -float("inf")
    for poem in candidates:
        inputs_cls = gen_tokenizer(poem, return_tensors="pt", truncation=True, padding="max_length", max_length=128).to(device)
        with torch.no_grad():
            logits = classifier(**inputs_cls).logits
            prob = torch.softmax(logits, dim=-1)[0, label2id[genre]].item()
        if prob > best_score:
            best_score = prob
            best_poem = poem

    return best_poem

print(f"Я могу сгенерировать для тебя стихотворение в любом из этих жанров: {', '.join(available_genres)}.")
genre_input = input("В каком жанре ты хочешь?\n").strip().lower()
if genre_input not in available_genres:
    print("Извини, я не знаю такого жанра. Используй один из предложенных.")
else:
    topic_input = input("Отлично! Тема стихотворения:\n").strip()
    print("\nГенерирую несколько вариантов и выбираю лучший по жанру...\n")
    poem = generate_4_lines_poem(genre_input, topic_input, n=5, max_new_tokens=250)
    print(f"\nСтихотворение ({genre_input}):\n{poem}")

## Вариант 2. Генерация с использованием Grok

In [ ]:
!pip install -q openai --upgrade

from google.colab import userdata
from openai import OpenAI
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
device = "cpu"
classifier = trainer.model.to(device)
classifier.eval()

available_genres = list(label2id.keys())

#Настройка клиента Grok
client = OpenAI(
    api_key  = userdata.get('XAI_API_KEY'),
    base_url = "https://api.x.ai/v1",
)

def generate_poem_with_grok(genre, topic, n=5, max_tokens=250):
    prompt = f"Сочини законченное стихотворение из 4 строк в жанре {genre} по теме {topic}:\n"
    candidates = []

    for _ in range(n):
        response = client.chat.completions.create(
            model="grok-3-mini",  #стабильная модель для Colab на март 2026
            messages=[
                {"role": "system", "content": "Ты остроумный и полезный помощник."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.9,
            max_tokens=max_tokens,
        )
        poem = response.choices[0].message.content.strip()
        #Иногда возвращается полный prompt, убираем его
        if poem.startswith(prompt):
            poem = poem[len(prompt):].strip()
        candidates.append(poem)

    #Выбор лучшего с помощью дообученной RuBERT
    best_poem = candidates[0] if candidates else ""
    best_score = -float("inf")

    for poem in candidates:
        inputs_cls = gen_tokenizer(poem, return_tensors="pt", truncation=True,
                                   padding="max_length", max_length=128).to(device)
        with torch.no_grad():
            logits = classifier(**inputs_cls).logits
            prob = torch.softmax(logits, dim=-1)[0, label2id[genre]].item()
        if prob > best_score:
            best_score = prob
            best_poem = poem

    return best_poem


print(f"Я могу сгенерировать для тебя стихотворение в любом из этих жанров: {', '.join(available_genres)}.")
genre_input = input("В каком жанре ты хочешь?\n").strip().lower()

if genre_input not in available_genres:
    print("Извини, я не знаю такого жанра. Используй один из предложенных.")
else:
    topic_input = input("Отлично! Тема стихотворения:\n").strip()
    print("\nГенерирую несколько вариантов через Grok и выбираю лучший по жанру...\n")
    poem = generate_poem_with_grok(genre_input, topic_input, n=5, max_tokens=250)
    print(f"\nСтихотворение ({genre_input}):\n{poem}")